In [1]:
#Load model directly
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "t5-small"

t5_tokenizer = AutoTokenizer.from_pretrained(model_name)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

## Getting baseline ROUGE Score

In [3]:
#Bring in test dataset (which we will make summaries for with untrained T5)
%store -r test_paired_summaries
%store -r val_paired_summaries

#Dictinory with the key being the text name and the value being a 3 element tuple of (text, summary, human-made flag) where human-made flag is true if the summary was made by a human



In [5]:
def summarize_story(story):
    prompt = "summarize: "

    # Tokenize the input text
    inputs = t5_tokenizer(prompt+story, max_length=1024, truncation=True, return_tensors="pt")

    # Dynamic max_new_tokens based on input length
    input_len = inputs["input_ids"].shape[1]

    #summary = 30% of input length, minimum of 30 tokens
    max_new = max(30, int(input_len * 0.3))

    # Generate the output sequence
    outputs = t5_model.generate(inputs["input_ids"], max_new_tokens=max_new)

    # Decode the output IDs back into human-readable text
    candidate = t5_tokenizer.batch_decode(outputs, skip_special_tokens=True, clean_up_tokenization_spaces=False)

    return candidate[0]

In [32]:
#dataset which contains scene or sonnet name as key and T5 summary as value
t5_test_sumaries=dict()
i=0

#Get summary of all stories in test dataset
for k, v in test_paired_summaries.items():
    story=v[0]
    t5_test_sumaries[k] = summarize_story(story)
    
    #for tracking progress of loop
    if i%10==0:
        print(i)
    i+=1


0
10
20
30
40
50
60
70
80
90
100
110
120
130
140


In [6]:
#dataset which contains scene or sonnet name as key and T5 summary as value
t5_val_sumaries=dict()
i=0

#Get summary of all stories in test dataset
for k, v in val_paired_summaries.items():
    story=v[0]
    t5_val_sumaries[k] = summarize_story(story)
    
    #for tracking progress of loop
    if i%10==0:
        print(i)
    i+=1


0
10
20
30
40
50
60
70
80
90
100
110
120
130


In [7]:
#Looking at data
#print(t5_test_sumaries)
print(t5_val_sumaries)

{'Hamlet-Act I-Scene I': "'Tis now struck twelve. Get thee to bed, Francisco. FRANCISCO. For this relief much thanks . 'Tis bitter cold, And I am sick at heart. BARNARDO. Give you good-night. MARCELLUS. O, farewell, honest soldier, who hath reliev’d you? HORATIO. 'tush, tush, 'twill not appear. BARNARDO. 'tush, tush, ", 'Hamlet-Act I-Scene II': 'hamlet, king, king, king, king, king, king, king, king, king, king, king, king, king, king, king, king, king, king, king, king, king, king, king, king, king, king, king, king, king, king, king, king, king, king, king, king, ', 'Hamlet-Act I-Scene III': 'a double blessing is a double grace; thou hast, and their adoption tried, Grapple them unto thy soul with hoops of steel; but not dull thy palm with entertainment . a double blessing is a double grace; thou . thou . thou . thou . thou . thou . thou . thou . thou . thou . thou .', 'Hamlet-Act I-Scene IV': 'hamlet, hamlet, is a nipping and an eager air . hamlet, hamlet, is a custom, hamlet, hamlet

In [34]:
#Store data
%store t5_test_sumaries

Stored 't5_test_sumaries' (dict)


In [8]:
#Store data
%store t5_val_sumaries

Stored 't5_val_sumaries' (dict)
